# 1 · LoRA in code

This notebook builds the model used in every LoRA experiment of the talk and checks it against the paper
(Hu et al., *LoRA: Low-Rank Adaptation of Large Language Models*, ICLR 2022, Sec. 4).

**Base model.** The same as in the VPT talk: ViT-B/16 pretrained with labels on ImageNet-21k
(`vit_base_patch16_224.orig_in21k` in timm), 85.8M parameters, 12 blocks of width d = 768.

**LoRA.** The base model stays frozen. Beside a chosen weight matrix W (d × d), LoRA trains two thin matrices,
A (r × d) and B (d × r), and adds their product to the layer's output:

  h = W x + (α / r) · B A x

B starts at zero, so training starts exactly from the pretrained model. As in the paper's default, LoRA goes beside
the query and value weights of every attention block, with rank r = 8 and α = r. Only the pairs and a linear
classification head are trained. After training, B A can be added into W once, and the model is again a plain ViT.

In [ ]:
import sys, json, time, math
from pathlib import Path
ROOT = Path.cwd().parent            # demo/lora
VPT = ROOT.parent / "vpt"           # data, linear-probe and full fine-tuning runs
sys.path.insert(0, str(ROOT))
import numpy as np, torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## Where the query and value weights are

timm computes query, key and value of a block with one fused 768 → 2304 matrix, `qkv`. Its first 768 output rows
are the query weight W_q, the next 768 the key, the last 768 the value W_v. LoRA has to update the first and last
thirds only.

In [ ]:
import timm
vit = timm.create_model("vit_base_patch16_224.orig_in21k", pretrained=True, num_classes=0)
n_params = sum(p.numel() for p in vit.parameters())
print(f"{n_params:,} parameters | width {vit.embed_dim} | blocks {len(vit.blocks)}")
print("block 1 attention:", vit.blocks[0].attn.qkv, "-> rows [0:768] query, [768:1536] key, [1536:2304] value")

## The LoRA layer

`lora_model.py` wraps each frozen `qkv` in a `LoRAQKV` module. Printed here is its forward pass: the frozen
output plus B(Ax) on the query and value slices.

In [ ]:
import inspect
from lora_model import LoRAQKV, LoRAViT, count, load_task
print(inspect.getsource(LoRAQKV.forward))

## Parameter counts against the paper

Each pair has r × d + d × r = 2dr numbers. With query and value in 12 blocks, d = 768 and r = 8:
2 × 768 × 8 × 2 × 12 = 294,912. The head comes on top (here the 7 classes of DermaMNIST).
For comparison, VPT-deep with 50 prompts per layer trained 460,800 numbers.

In [ ]:
base = LoRAViT(7, rank=0, vit=vit)
print(f"{'':22s}{'pairs':>10s}{'head':>8s}{'trainable':>12s}{'% of backbone':>15s}")
for r in (1, 2, 4, 8, 16, 64):
    c = count(LoRAViT(7, rank=r, vit=vit))
    print(f"{'LoRA r = ' + str(r):22s}{c['lora']:>10,}{c['head']:>8,}{c['trainable']:>12,}{c['trainable_pct']:>14.3f}%")
    assert c["lora"] == 2 * 768 * r * 2 * 12

## At step zero, LoRA is the pretrained model

B is zero at initialisation, so B A x = 0 and the features are exactly those of the frozen base model.

In [ ]:
torch.manual_seed(0)
x = torch.randn(2, 3, 224, 224)
with torch.no_grad():
    plain = LoRAViT(7, rank=0, vit=vit).eval().forward_features(x)
    m = LoRAViT(7, rank=8, vit=vit).eval()
    print("largest feature difference at init:", (m.forward_features(x) - plain).abs().max().item())

## Only the pairs and the head learn

One training step with a random batch: the frozen weights receive no gradient, the A and B matrices of every block
do. The gradient still flows *through* every frozen layer to reach them, which is why LoRA's training memory on a
ViT is not much lower than full fine-tuning's (notebook 3).

In [ ]:
m = LoRAViT(7, rank=8, vit=vit).train()
loss = torch.nn.functional.cross_entropy(m(torch.randn(4, 3, 224, 224)), torch.randint(0, 7, (4,)))
loss.backward()
frozen = sum(1 for n, p in m.vit.named_parameters() if p.grad is not None and not n.endswith((".A", ".B")))
print("frozen tensors that received a gradient:", frozen)
print("B gradient per block:", [f"{l.B.grad.abs().sum():.1e}" for l in m.lora_layers()])
print("A gradient per block (zero at the first step, because B = 0):", [f"{l.A.grad.abs().sum():.0e}" for l in m.lora_layers()][:4], "...")
m.zero_grad()

## Merging: the same outputs, the plain model's size and speed

With trained-looking B matrices (random here), `merge()` adds (α / r) B A into each `qkv` weight and removes the
bypass. The outputs agree up to float rounding, and the parameter count is back to the base model's.

In [ ]:
import copy
m = LoRAViT(7, rank=8, vit=copy.deepcopy(vit)).eval()     # a copy, because merging changes the frozen weights
for l in m.lora_layers():
    l.B.data.normal_(0, 0.02)
torch.set_num_threads(4)
with torch.no_grad():
    def ms(f, n=5):
        f(); t0 = time.perf_counter()
        for _ in range(n): f()
        return (time.perf_counter() - t0) / n * 1000
    before, t_before = m(x), ms(lambda: m(x[:1]))
    m.merge()
    after, t_after = m(x), ms(lambda: m(x[:1]))
print("largest output difference after merging:", (after - before).abs().max().item())
print("parameters after merging:", f"{sum(p.numel() for p in m.vit.parameters()):,}", "(base model:", f"{n_params:,})")
print(f"CPU time per image: {t_before:.0f} ms with the bypass, {t_after:.0f} ms merged")